# LOINC Mapping in LCED Lab Data

**LOINC** (Logical Observation Identifiers Names and Codes) is the standard coding system for laboratory and clinical observations. In IBM LCED, two sources carry LOINC codes:

| Source | Column | Domain |
|---|---|---|
| `v_observation` (Explorys EMR) | `loinc_test_id` | Structured EMR lab values |
| `v_lab_results` (MarketScan claims) | `loinccd` | Claims-based lab results |

This notebook shows how to search for clinical concepts by LOINC, filter lab results, and join across sources — translating the original R `labtest code.R` workflow.

In [1]:
import numpy as np
import pandas as pd

from ehrdata.io.source.adapters import lced
from ehrdata.io.source.vocab import loinc as loinc_vocab

## 1. LOINC Reference Data

The `loinc.load_loinc_map` function reads a LOINC reference table and returns a DataFrame with at minimum `loinc_code` and `long_common_name` columns.

The original R code queried this from the database via:
```r
loinc.df <- dbGetQuery(conn, "SELECT * FROM xref.loinc")
```

In Python we provide it as a CSV file. Below we build a representative subset covering the most common clinical concepts in diabetes research.

In [2]:
# Representative LOINC codes used in T2D / cardiometabolic research
# In practice, load from loinc_map.csv (subset of the full LOINC table)
loinc_reference = pd.DataFrame({
    "loinc_code": [
        "4548-4",  "17856-6", "59261-8",  # HbA1c
        "29463-7", "3141-9",  "75292-3",  # Weight
        "8302-2",  "3137-7",  "8308-9",   # Height
        "39156-5", "59574-4", "89270-3",  # BMI
        "2345-7",  "14749-6",             # Glucose
        "2160-0",  "38483-4",             # Creatinine
        "4544-3",  "20570-8",             # Haematocrit
        "718-7",   "26515-7",             # Haemoglobin, Platelets
    ],
    "long_common_name": [
        "Hemoglobin A1c/Hemoglobin.total in Blood",
        "Hemoglobin A1c/Hemoglobin.total in Blood by HPLC",
        "Hemoglobin A1c/Hemoglobin.total in Blood by Immunoassay",
        "Body weight",
        "Body weight Measured",
        "Body weight - Reported",
        "Body height",
        "Body height Measured",
        "Body height - standing",
        "Body mass index (BMI) [Ratio]",
        "Body mass index (BMI) [Percentile]",
        "Body mass index (BMI) [Ratio] Reported",
        "Glucose [Mass/volume] in Serum or Plasma",
        "Glucose [Moles/volume] in Serum or Plasma",
        "Creatinine [Mass/volume] in Serum or Plasma",
        "Creatinine [Mass/volume] in Blood",
        "Hematocrit [Volume Fraction] of Blood by Automated count",
        "Hematocrit [Volume Fraction] of Blood",
        "Hemoglobin [Mass/volume] in Blood",
        "Platelets [#/volume] in Blood by Automated count",
    ],
})

print(f"Reference LOINC codes: {len(loinc_reference)}")
loinc_reference.head(8)

Reference LOINC codes: 20


,loinc_code,long_common_name
0,4548-4,Hemoglobin A1c/Hemoglobin.total in Blood
1,17856-6,Hemoglobin A1c/Hemoglobin.total in Blood by HPLC
2,59261-8,Hemoglobin A1c/Hemoglobin.total in Blood by Im...
3,29463-7,Body weight
4,3141-9,Body weight Measured
5,75292-3,Body weight - Reported
6,8302-2,Body height
7,3137-7,Body height Measured


## 2. Concept Search

The R pipeline used `grep` to search concept names. Python's `str.contains` is the direct equivalent.

In [3]:
def search_loinc(df, pattern, col="long_common_name"):
    """Case-insensitive substring search over LOINC names."""
    mask = df[col].str.contains(pattern, case=False, na=False)
    return df[mask].reset_index(drop=True)


# --- HbA1c ---
print("=== HbA1c ===")
print(search_loinc(loinc_reference, r"hba1c|hemoglobin a1c|haemoglobin a1c")[["loinc_code", "long_common_name"]])
print()

# --- Weight ---
print("=== Body Weight ===")
print(search_loinc(loinc_reference, r"body weight")[["loinc_code", "long_common_name"]])
print()

# --- Height ---
print("=== Body Height ===")
print(search_loinc(loinc_reference, r"body height")[["loinc_code", "long_common_name"]])
print()

# --- BMI ---
print("=== BMI ===")
print(search_loinc(loinc_reference, r"body mass index|\bbmi\b")[["loinc_code", "long_common_name"]])

=== HbA1c ===
  loinc_code                                   long_common_name
0     4548-4           Hemoglobin A1c/Hemoglobin.total in Blood
1    17856-6   Hemoglobin A1c/Hemoglobin.total in Blood by HPLC
2    59261-8  Hemoglobin A1c/Hemoglobin.total in Blood by Im...

=== Body Weight ===
  loinc_code        long_common_name
0    29463-7             Body weight
1     3141-9    Body weight Measured
2    75292-3  Body weight - Reported

=== Body Height ===
  loinc_code        long_common_name
0     8302-2             Body height
1     3137-7    Body height Measured
2     8308-9  Body height - standing

=== BMI ===
  loinc_code                        long_common_name
0    39156-5           Body mass index (BMI) [Ratio]
1    59574-4      Body mass index (BMI) [Percentile]
2    89270-3  Body mass index (BMI) [Ratio] Reported


## 3. Key LOINC Codes for Diabetes Research

Based on the original R analysis, the following LOINC codes are used most frequently:

| Concept | Primary LOINC | Common variants |
|---|---|---|
| HbA1c | **4548-4** | 17856-6, 59261-8 |
| Body weight | **29463-7** | 3141-9, 75292-3 |
| Body height | **8302-2** | 3137-7, 8308-9 |
| BMI | **39156-5** | 59574-4, 89270-3 |

> **Tip**: When filtering `v_observation`, use all known variants of a concept to maximise recall. `4548-4` alone captures ~80 % of HbA1c results in most EMR datasets.

In [4]:
# Canonical concept code lists (union of variants)
HBA1C_LOINCS  = ["4548-4", "17856-6", "59261-8"]
WEIGHT_LOINCS = ["29463-7", "3141-9", "75292-3"]
HEIGHT_LOINCS = ["8302-2", "3137-7", "8308-9"]
BMI_LOINCS    = ["39156-5", "59574-4", "89270-3"]

all_concept_loincs = {
    "hba1c":  HBA1C_LOINCS,
    "weight": WEIGHT_LOINCS,
    "height": HEIGHT_LOINCS,
    "bmi":    BMI_LOINCS,
}
for concept, codes in all_concept_loincs.items():
    print(f"{concept:8s}: {codes}")

hba1c   : ['4548-4', '17856-6', '59261-8']
weight  : ['29463-7', '3141-9', '75292-3']
height  : ['8302-2', '3137-7', '8308-9']
bmi     : ['39156-5', '59574-4', '89270-3']


## 4. Synthetic LCED Lab Data

Below we simulate the two lab sources in LCED so you can follow the filtering steps without raw data access.

In [5]:
np.random.seed(42)
n = 30

patient_pool = [f"P{i:04d}" for i in range(1, 11)]

# Explorys v_observation (EMR)
v_observation = pd.DataFrame({
    "patient_id":       np.random.choice(patient_pool, n),
    "observation_date": pd.date_range("2014-01-01", periods=n, freq="14D"),
    "loinc_test_id":    np.random.choice(
        HBA1C_LOINCS + WEIGHT_LOINCS + HEIGHT_LOINCS + BMI_LOINCS + ["2345-7", "2160-0"],
        n,
    ),
    "std_value":  np.round(np.random.uniform(4.0, 12.0, n), 1),
    "std_uom":    np.random.choice(["%", "kg", "cm", "kg/m2", "mg/dL"], n),
})

# MarketScan v_lab_results (claims)
n2 = 20
v_lab_results = pd.DataFrame({
    "patient_id":  np.random.choice(patient_pool, n2),
    "svcdate":     pd.date_range("2014-03-01", periods=n2, freq="21D"),
    "loinccd":     np.random.choice(HBA1C_LOINCS + WEIGHT_LOINCS + ["2345-7"], n2),
    "result":      np.round(np.random.uniform(4.0, 12.0, n2), 1),
    "resunit":     np.random.choice(["%", "kg", "mg/dL"], n2),
    "resltcat":    np.random.choice([None, "Normal", "High", "Low"], n2),
})

print(f"v_observation rows : {len(v_observation)}")
print(f"v_lab_results rows : {len(v_lab_results)}")

v_observation rows : 30
v_lab_results rows : 20


## 5. Building the Canonical Lab Table

`lced.build_labtest` unifies both sources into a single canonical schema:

| Column | Source | Notes |
|---|---|---|
| `patient_id` | both | |
| `eventdate` | both | `observation_date` or `svcdate` |
| `value` | both | `std_value` or `result` |
| `unit` | both | `std_uom` or `resunit` |
| `valuecat` | v_lab_results | `resltcat` (only 3 % non-missing) |
| `loinc` | both | `loinc_test_id` or `loinccd` |

In [6]:
labs = lced.build_labtest(v_observation, v_lab_results)
print(f"Combined lab rows: {len(labs)}")
labs.head(6)

Combined lab rows: 50


,patient_id,eventdate,value,valuecat,unit,loinc
0,P0001,2014-04-12,6.0,Normal,%,2345-7
1,P0001,2014-10-22,6.2,None,cm,89270-3
2,P0001,2014-12-17,7.4,None,cm,3137-7
3,P0002,2014-08-13,10.0,None,cm,39156-5
4,P0002,2014-09-24,11.7,None,cm,29463-7
5,P0002,2014-11-08,5.2,High,%,4548-4


## 6. Filtering by LOINC Concept

With the canonical `loinc` column present, filtering to a specific concept is a single boolean mask — equivalent to the R `WHERE a.loinc_test_id IN (...)` query.

In [7]:
# HbA1c results
hba1c_labs = labs[labs["loinc"].isin(HBA1C_LOINCS)].copy()
hba1c_labs["value"] = pd.to_numeric(hba1c_labs["value"], errors="coerce")

print(f"HbA1c observations: {len(hba1c_labs)}")
print(f"Patients with HbA1c: {hba1c_labs['patient_id'].nunique()}")
print()
print(hba1c_labs[["patient_id", "eventdate", "value", "unit", "loinc"]].head(8))

HbA1c observations: 17
Patients with HbA1c: 8

   patient_id  eventdate  value   unit    loinc
5       P0002 2014-11-08    5.2      %   4548-4
8       P0003 2014-07-02    8.2  mg/dL  17856-6
11      P0004 2014-01-15   10.0     cm  59261-8
14      P0004 2014-05-24    5.8     kg   4548-4
17      P0004 2015-02-21    4.3  mg/dL   4548-4
18      P0005 2014-02-12    5.7     cm  59261-8
20      P0005 2014-06-14    8.5  mg/dL   4548-4
25      P0006 2014-09-10    8.7      %  17856-6


## 7. Poor Glycaemic Control

The original R cohort study defined poor glycaemic control as HbA1c ≥ 7 % (LOINC 4548-4 and variants), with at least one of:
- A value ≥ 9 %, **or**
- Two consecutive values ≤ 183 days apart

The Python translation uses `groupby` + `diff` instead of the R `aggregate(difftime(...))` loop.

In [8]:
a1c = (
    hba1c_labs
    .dropna(subset=["value"])
    .query("value >= 7")
    .sort_values(["patient_id", "eventdate"])
    .reset_index(drop=True)
)

# Days between consecutive HbA1c measurements per patient
a1c["prev_date"] = a1c.groupby("patient_id")["eventdate"].shift(1)
a1c["days_since_last"] = (a1c["eventdate"] - a1c["prev_date"]).dt.days

# Poor control: HbA1c ≥ 9  OR  interval ≤ 183 days
a1c["poor_control"] = (a1c["value"] >= 9) | (a1c["days_since_last"] <= 183)

poor_ctrl = a1c[a1c["poor_control"]]
print(f"Poor-control HbA1c events : {len(poor_ctrl)}")
print(f"Patients flagged           : {poor_ctrl['patient_id'].nunique()}")
poor_ctrl[["patient_id", "eventdate", "value", "days_since_last", "poor_control"]].head(8)

Poor-control HbA1c events : 4
Patients flagged           : 3


,patient_id,eventdate,value,days_since_last,poor_control
1,P0004,2014-01-15,10.0,NaN,True
4,P0006,2015-01-31,7.3,143.0,True
7,P0010,2014-11-29,12.0,NaN,True
8,P0010,2014-12-31,7.2,32.0,True


## 8. Per-Patient LOINC Coverage

A useful QC check is verifying which patients have observations for each concept category.

In [9]:
def concept_flag(df, loinc_list, concept_name):
    return (
        df[df["loinc"].isin(loinc_list)]
        .groupby("patient_id")
        .size()
        .rename(concept_name)
        .gt(0)  # boolean: has any measurement
    )

coverage = pd.concat([
    concept_flag(labs, HBA1C_LOINCS,  "hba1c"),
    concept_flag(labs, WEIGHT_LOINCS, "weight"),
    concept_flag(labs, HEIGHT_LOINCS, "height"),
    concept_flag(labs, BMI_LOINCS,    "bmi"),
], axis=1).fillna(False)

print("Concept coverage per patient:")
print(coverage)
print()
print("Overall coverage rates:")
print(coverage.mean().round(3))

Concept coverage per patient:
            hba1c  weight  height    bmi
patient_id                              
P0002        True    True   False   True
P0003        True    True    True  False
P0004        True    True    True  False
P0005        True    True    True   True
P0006        True    True   False   True
P0007        True    True    True  False
P0008        True    True   False   True
P0010        True    True    True  False
P0001       False   False    True   True

Overall coverage rates:
hba1c     0.889
weight    0.889
height    0.667
bmi       0.556
dtype: float64


/var/folders/cz/cdvvmc8d6mn_kk380srmqfl00000gn/T/ipykernel_72177/3138325945.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ], axis=1).fillna(False)


## Summary

| Task | Python API |
|---|---|
| Search LOINC by name | `df["long_common_name"].str.contains(pattern, case=False)` |
| Load LOINC vocab | `loinc.load_loinc_map("loinc_map.csv")` |
| Build canonical lab table | `lced.build_labtest(v_observation, v_lab_results)` |
| Filter by concept | `labs[labs["loinc"].isin(HBA1C_LOINCS)]` |
| Poor glycaemic control flag | `groupby` + `shift` + `diff` on sorted dates |

**Next steps**: See `source_lced_cohort.ipynb` for a complete T2D second-line therapy cohort study that uses these HbA1c filters.